In [3]:
from pathlib import Path
import json, pandas as pd, matplotlib.pyplot as plt

path = "experiment_results.json"
with open(path) as f:
    data = json.load(f)

rows = []
for r in data:
    p = r["params"]
    m = r["metrics"]
    topk = p["top_k"]
    rows.append({
        "embedding": p["embedding_model"].split("/")[-1],
        "retriever": p["retriever_type"],
        "top_k": topk,
        "recall": next(v for k,v in m.items() if k.startswith("recall@")),
        "mrr": m["mrr"],
        "ndcg": next(v for k,v in m.items() if k.startswith("ndcg@")),
        "latency": r["latency"]
    })

df = pd.DataFrame(rows)

best = df[df["retriever"]=="vector"]

plt.figure(figsize=(8,5))
for emb, g in best.groupby("embedding"):
    g = g.sort_values("top_k")
    plt.plot(g["top_k"], g["recall"], marker="o", label=emb)
plt.title("Recall vs Top-K (Vector Retriever)")
plt.xlabel("Top-K")
plt.ylabel("Recall")
plt.legend()
out1="rag_recall_comparison.png"
plt.savefig(out1, bbox_inches="tight")
plt.close()

# best configuration by ndcg
top = df.sort_values("ndcg", ascending=False).head(10)
plt.figure(figsize=(10,5))
labels=[f"{r.embedding}\n{r.retriever}\nK={r.top_k}" for _,r in top.iterrows()]
plt.bar(range(len(top)), top["ndcg"])
plt.xticks(range(len(top)), labels, rotation=45, ha="right")
plt.ylabel("NDCG")
plt.title("Top 10 Retrieval Configurations")
plt.tight_layout()
out2="rag_top10_ndcg.png"
plt.savefig(out2)
plt.close()

print({"recall_chart":out1,"top10_chart":out2})


{'recall_chart': 'rag_recall_comparison.png', 'top10_chart': 'rag_top10_ndcg.png'}
